# 05 — Multi-Head Merge (Flow + Heat + Species)

Trains a single **MultiHeadMGN** with a shared encoder and three decoder heads:
- **Flow head** → Ux, Uy, Uz, p
- **Heat head** → T
- **Species head** → TMA

**Transfer init:** Shared encoder + heat decoder from `heat_head_final.pt`. Species decoder from `species_head_final.pt`. Flow decoder trained fresh.

**Masked losses:** loss mask per case — all 6 fields active for our 80 reactingFoam cases.

**Requires:** Colab A100 GPU

## 0. Setup

In [ ]:
import subprocess, sys, os

subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy', 'tqdm'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup done.')

## 1. Config

In [ ]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), 'Switch to A100 GPU runtime first'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

DRIVE_BASE   = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_DIR     = DRIVE_BASE / 'showerhead_openfoam'
CKPT_DIR     = DRIVE_BASE / 'checkpoints' / 'multihead'
HEAT_CKPT    = DRIVE_BASE / 'checkpoints' / 'heat_head'    / 'heat_head_final.pt'
SPECIES_CKPT = DRIVE_BASE / 'checkpoints' / 'species_head' / 'species_head_final.pt'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

h5_files = sorted(HDF5_DIR.glob('case_*.h5'))
print(f'Found {len(h5_files)} HDF5 files')
assert len(h5_files) > 0

# Field layout in outputs/node_fields [N, 6]
OUTPUT_COLS = ['Ux', 'Uy', 'Uz', 'p', 'T', 'TMA']
FLOW_IDXS   = [0, 1, 2, 3]
HEAT_IDX    = 4
SPECIES_IDX = 5

CFG = {
    'node_input_dim':  22,
    'edge_input_dim':  4,
    'hidden_dim':      128,
    'n_layers':        8,
    'flow_out_dim':    4,
    'heat_out_dim':    1,
    'species_out_dim': 1,
    'k_neighbors':     6,
    'train_frac':      0.875,
    'epochs':          60,
    'lr':              1e-4,
    'grad_accum':      4,
    'grad_clip':       1.0,
    'bf16':            True,
    'save_every':      20,
    'log_every':       5,
    'w_flow':          1.0,
    'w_heat':          1.0,
    'w_species':       3.0,
}
print('Config ready.')

## 2. Load HDF5 data (all 6 output fields)

In [ ]:
import h5py
import numpy as np
from tqdm import tqdm
import random

def load_case(h5_path):
    with h5py.File(h5_path, 'r') as f:
        coords       = f['coords'][:]
        node_feats   = f['inputs/node_features'][:]
        global_feats = f['inputs/global'][:]
        out_fields   = f['outputs/node_fields'][:]

    N = len(coords)
    global_broadcast = np.tile(global_feats, (N, 1)).astype(np.float32)
    node_input = np.concatenate([node_feats, global_broadcast], axis=1)

    T   = out_fields[:, HEAT_IDX]
    TMA = out_fields[:, SPECIES_IDX]

    if T.std() < 0.01:
        return None

    return {
        'coords':     coords.astype(np.float32),
        'node_input': node_input,
        'outputs':    out_fields.astype(np.float32),
        'mask':       {'flow': True, 'heat': True, 'species': bool(TMA.max() > 1e-5)},
        'name':       h5_path.stem,
    }

print('Loading cases...')
all_cases = []
for p in tqdm(h5_files):
    try:
        c = load_case(p)
        if c is not None:
            all_cases.append(c)
    except Exception as e:
        print(f'  skip {p.name}: {e}')

print(f'Loaded {len(all_cases)} valid cases')
species_active = sum(1 for c in all_cases if c['mask']['species'])
print(f'  species head active in {species_active}/{len(all_cases)} cases')

random.seed(42)
random.shuffle(all_cases)
n_train = int(len(all_cases) * CFG['train_frac'])
train_cases = all_cases[:n_train]
val_cases   = all_cases[n_train:]
print(f'Train: {len(train_cases)}   Val: {len(val_cases)}')

c = train_cases[0]
print(f"\nSample: {c['name']}  N={len(c['coords']):,}")
for i, col in enumerate(OUTPUT_COLS):
    v = c['outputs'][:, i]
    print(f'  {col:4s}: {v.min():.3e} – {v.max():.3e}')

## 3. Normalisation stats (per output field)

In [ ]:
norm_path = CKPT_DIR / 'multihead_norm_stats.npz'

if norm_path.exists():
    stats     = np.load(norm_path)
    node_mean = stats['node_mean']; node_std = stats['node_std']
    out_mean  = stats['out_mean'];  out_std  = stats['out_std']
    print('Loaded norm stats from checkpoint.')
else:
    sample    = train_cases[:20]
    all_nodes = np.concatenate([c['node_input'] for c in sample], axis=0)
    all_outs  = np.concatenate([c['outputs']    for c in sample], axis=0)

    node_mean = all_nodes.mean(0).astype(np.float32)
    node_std  = np.clip(all_nodes.std(0), 1e-6, None).astype(np.float32)
    out_mean  = all_outs.mean(0).astype(np.float32)
    out_std   = np.clip(all_outs.std(0), 1e-8, None).astype(np.float32)

    np.savez(norm_path, node_mean=node_mean, node_std=node_std,
             out_mean=out_mean, out_std=out_std)
    print('Computed and saved norm stats.')

print('Output normalisation:')
for i, col in enumerate(OUTPUT_COLS):
    print(f'  {col:4s}  mean={out_mean[i]:.3e}  std={out_std[i]:.3e}')

## 4. Graph builder

In [ ]:
from scipy.spatial import cKDTree

K = CFG['k_neighbors']

def make_graph(case):
    coords     = case['coords']
    node_input = case['node_input']
    outputs    = case['outputs']
    N          = len(coords)

    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K + 1)
    idx = idx[:, 1:]

    src = np.repeat(np.arange(N), K)
    dst = idx.flatten()

    diff = coords[dst] - coords[src]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)
    med  = float(np.median(dist)) + 1e-8
    edge_feats = np.concatenate([diff / med, dist / med], axis=1).astype(np.float32)

    node_norm = ((node_input - node_mean) / node_std).astype(np.float32)
    out_norm  = ((outputs - out_mean) / out_std).astype(np.float32)

    return {
        'x':          torch.from_numpy(node_norm),
        'edge_index': torch.tensor(np.stack([src, dst]), dtype=torch.long),
        'edge_attr':  torch.from_numpy(edge_feats),
        'y':          torch.from_numpy(out_norm),
        'mask':       case['mask'],
    }

g = make_graph(train_cases[0])
print(f'x: {tuple(g["x"].shape)}  edge_index: {tuple(g["edge_index"].shape)}  y: {tuple(g["y"].shape)}')

## 5. MultiHeadMGN model

In [ ]:
import torch.nn as nn
from torch_geometric.nn import MessagePassing

def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)


class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3 * hidden, hidden)
        self.node_mlp  = mlp(2 * hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        N     = x.size(0)
        new_e = self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], dim=-1))
        new_e = self.edge_norm(new_e + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(N, N))
        new_x = self.node_mlp(torch.cat([x, agg], dim=-1))
        new_x = self.node_norm(new_x + x)
        return new_x, new_e

    def message(self, edge_attr):
        return edge_attr


class MultiHeadMGN(nn.Module):
    """Shared encoder + flow / heat / species decoder heads."""

    def __init__(self, node_dim, edge_dim,
                 flow_out=4, heat_out=1, species_out=1,
                 hidden=128, n_layers=8):
        super().__init__()
        self.node_enc   = mlp(node_dim, hidden)
        self.edge_enc   = mlp(edge_dim, hidden)
        self.processors = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)

    def encode(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return x

    def forward(self, x, edge_index, edge_attr):
        latent = self.encode(x, edge_index, edge_attr)
        return self.flow_dec(latent), self.heat_dec(latent), self.species_dec(latent)


model = MultiHeadMGN(
    node_dim    = CFG['node_input_dim'],
    edge_dim    = CFG['edge_input_dim'],
    flow_out    = CFG['flow_out_dim'],
    heat_out    = CFG['heat_out_dim'],
    species_out = CFG['species_out_dim'],
    hidden      = CFG['hidden_dim'],
    n_layers    = CFG['n_layers'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'MultiHeadMGN: {n_params/1e6:.2f}M parameters')

## 6. Transfer weight initialisation

Copy shared encoder + heat decoder from `heat_head_final.pt`.  
Copy species decoder from `species_head_final.pt`.  
Flow decoder starts from random init.

In [ ]:
def transfer_weights(src_state, dst_module, prefix):
    dst_state = dst_module.state_dict()
    loaded = 0
    for k, v in src_state.items():
        if k.startswith(prefix):
            new_k = k[len(prefix):]
            if new_k in dst_state and dst_state[new_k].shape == v.shape:
                dst_state[new_k] = v
                loaded += 1
    dst_module.load_state_dict(dst_state, strict=False)
    return loaded


if HEAT_CKPT.exists():
    heat_sd = torch.load(HEAT_CKPT, map_location='cpu')['model']
    for name in ('node_enc', 'edge_enc', 'processors'):
        n = transfer_weights(heat_sd, getattr(model, name), f'{name}.')
        print(f'  {name}: {n} tensors transferred')
    n = transfer_weights(heat_sd, model.heat_dec, 'decoder.')
    print(f'  heat_dec: {n} tensors transferred')
    print(f'Encoder + heat decoder initialised from {HEAT_CKPT.name}')
else:
    print(f'WARNING: {HEAT_CKPT} not found — random init')

if SPECIES_CKPT.exists():
    sp_sd = torch.load(SPECIES_CKPT, map_location='cpu')['model']
    n = transfer_weights(sp_sd, model.species_dec, 'decoder.')
    print(f'  species_dec: {n} tensors transferred from {SPECIES_CKPT.name}')
else:
    print(f'WARNING: {SPECIES_CKPT} not found — random init')

## 7. Training with masked per-head losses

In [ ]:
import time

resume_ckpt = sorted(CKPT_DIR.glob('multihead_epoch*.pt'))
start_epoch = 0
train_losses, val_losses = [], []
head_train = {'flow': [], 'heat': [], 'species': []}

if resume_ckpt:
    ckpt = torch.load(resume_ckpt[-1], map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    start_epoch  = ckpt['epoch'] + 1
    train_losses = ckpt.get('train_losses', [])
    val_losses   = ckpt.get('val_losses', [])
    head_train   = ckpt.get('head_train', {'flow': [], 'heat': [], 'species': []})
    print(f'Resumed from epoch {start_epoch}')
else:
    print('Training from scratch (with pretrained init).')

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-5
)
for _ in range(start_epoch):
    scheduler.step()

mse   = nn.functional.mse_loss
W     = {'flow': CFG['w_flow'], 'heat': CFG['w_heat'], 'species': CFG['w_species']}
dtype = torch.bfloat16 if CFG['bf16'] else torch.float32
G     = CFG['grad_accum']

print(f'\nTraining {CFG["epochs"]-start_epoch} more epochs  '
      f'weights: flow={W["flow"]} heat={W["heat"]} species={W["species"]}')
t0 = time.time()

for epoch in range(start_epoch, CFG['epochs']):
    model.train()
    random.shuffle(train_cases)
    ep_loss = 0.0
    ep_h = {'flow': 0.0, 'heat': 0.0, 'species': 0.0}
    ep_n = {'flow': 0,   'heat': 0,   'species': 0}
    optimizer.zero_grad()

    for step, case in enumerate(train_cases):
        g = make_graph(case)
        x  = g['x'].to(DEVICE)
        ei = g['edge_index'].to(DEVICE)
        ea = g['edge_attr'].to(DEVICE)
        y  = g['y'].to(DEVICE)
        mk = g['mask']

        with torch.amp.autocast('cuda', dtype=dtype):
            fp, hp, sp = model(x, ei, ea)
            total = torch.tensor(0.0, device=DEVICE)
            if mk['flow']:
                lf = mse(fp, y[:, FLOW_IDXS])
                total = total + W['flow'] * lf
                ep_h['flow'] += lf.item(); ep_n['flow'] += 1
            if mk['heat']:
                lh = mse(hp, y[:, HEAT_IDX:HEAT_IDX+1])
                total = total + W['heat'] * lh
                ep_h['heat'] += lh.item(); ep_n['heat'] += 1
            if mk['species']:
                ls = mse(sp, y[:, SPECIES_IDX:SPECIES_IDX+1])
                total = total + W['species'] * ls
                ep_h['species'] += ls.item(); ep_n['species'] += 1
            (total / G).backward()

        if (step + 1) % G == 0 or step == len(train_cases) - 1:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += total.item()
        del x, ei, ea, y, fp, hp, sp, total
        torch.cuda.empty_cache()

    scheduler.step()
    train_losses.append(ep_loss / len(train_cases))
    for k in ep_h:
        head_train[k].append(ep_h[k] / max(ep_n[k], 1))

    if (epoch + 1) % CFG['log_every'] == 0 or epoch == start_epoch:
        model.eval()
        vl_h = {'flow': 0.0, 'heat': 0.0, 'species': 0.0}
        vl_n = {'flow': 0,   'heat': 0,   'species': 0}
        with torch.no_grad():
            for case in val_cases:
                g  = make_graph(case)
                fp, hp, sp = model(g['x'].to(DEVICE), g['edge_index'].to(DEVICE),
                                   g['edge_attr'].to(DEVICE))
                y  = g['y'].to(DEVICE)
                mk = g['mask']
                if mk['flow']:    vl_h['flow']    += mse(fp, y[:, FLOW_IDXS]).item();              vl_n['flow']    += 1
                if mk['heat']:    vl_h['heat']    += mse(hp, y[:, HEAT_IDX:HEAT_IDX+1]).item();   vl_n['heat']    += 1
                if mk['species']: vl_h['species'] += mse(sp, y[:, SPECIES_IDX:SPECIES_IDX+1]).item(); vl_n['species'] += 1
                del fp, hp, sp, y
                torch.cuda.empty_cache()
        vm = {k: vl_h[k] / max(vl_n[k], 1) for k in vl_h}
        val_losses.append(sum(vm.values()) / 3)
        elapsed = (time.time() - t0) / 60
        print(f'Ep {epoch+1:3d}/{CFG["epochs"]}  '
              f'flow={vm["flow"]:.4f}  heat={vm["heat"]:.4f}  species={vm["species"]:.4f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  [{elapsed:.1f}min]')

    if (epoch + 1) % CFG['save_every'] == 0:
        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
            'head_train': head_train,
        }, CKPT_DIR / f'multihead_epoch{epoch+1:04d}.pt')
        print(f'  -> Checkpoint saved (epoch {epoch+1})')

print(f'\nTraining complete in {(time.time()-t0)/60:.1f} min')

## 8. Loss curves + multi-panel prediction plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 0: per-head train losses
for ax, head, label in zip(axes[0],
                            ['flow', 'heat', 'species'],
                            ['Flow (Ux/Uy/Uz/p)', 'Heat (T)', 'Species (TMA)']):
    ax.plot(head_train[head], alpha=0.8)
    ax.set_yscale('log')
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (train)')

# Row 1: pred vs CFD scatter for one val case
model.eval()
case = val_cases[0]
g = make_graph(case)
with torch.no_grad():
    fp, hp, sp = model(g['x'].to(DEVICE), g['edge_index'].to(DEVICE),
                       g['edge_attr'].to(DEVICE))
    fp = fp.cpu().numpy()
    hp = hp.cpu().numpy().flatten()
    sp = sp.cpu().numpy().flatten()

T_pred   = hp * out_std[HEAT_IDX]    + out_mean[HEAT_IDX]
T_true   = case['outputs'][:, HEAT_IDX]
TMA_pred = sp * out_std[SPECIES_IDX] + out_mean[SPECIES_IDX]
TMA_true = case['outputs'][:, SPECIES_IDX]
# fp[:,4] = [Ux, Uy, Uz, p] — take only first 3 columns for velocity magnitude
U_pred   = np.linalg.norm(fp[:, :3] * out_std[:3] + out_mean[:3], axis=1)
U_true   = np.linalg.norm(case['outputs'][:, :3], axis=1)

z = case['coords'][:, 2]
r_all = np.sqrt(case['coords'][:, 0]**2 + case['coords'][:, 1]**2)

# T at wafer (z < 3mm)
wm = z < 0.003
r  = r_all[wm]; so = np.argsort(r)
axes[1, 0].scatter(r[so]*1e3, T_true[wm][so], s=1, alpha=0.3, label='CFD')
axes[1, 0].scatter(r[so]*1e3, T_pred[wm][so], s=1, alpha=0.3, label='Pred')
axes[1, 0].set_xlabel('r [mm]'); axes[1, 0].set_ylabel('T [K]')
axes[1, 0].set_title('T at wafer'); axes[1, 0].legend(markerscale=5)

# TMA at standoff mid-plane (5-15mm)
sm = (z > 0.005) & (z < 0.015)
r2 = r_all[sm]; so2 = np.argsort(r2)
axes[1, 1].scatter(r2[so2]*1e3, TMA_true[sm][so2], s=1, alpha=0.3, label='CFD')
axes[1, 1].scatter(r2[so2]*1e3, TMA_pred[sm][so2], s=1, alpha=0.3, label='Pred')
axes[1, 1].set_xlabel('r [mm]'); axes[1, 1].set_ylabel('TMA')
axes[1, 1].set_title('TMA at standoff mid-plane'); axes[1, 1].legend(markerscale=5)

# |U| at standoff mid-plane
axes[1, 2].scatter(r2[so2]*1e3, U_true[sm][so2], s=1, alpha=0.3, label='CFD')
axes[1, 2].scatter(r2[so2]*1e3, U_pred[sm][so2], s=1, alpha=0.3, label='Pred')
axes[1, 2].set_xlabel('r [mm]'); axes[1, 2].set_ylabel('|U| [m/s]')
axes[1, 2].set_title('|U| at standoff mid-plane'); axes[1, 2].legend(markerscale=5)

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'multihead_results.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'Case: {case["name"]}')

## 9. Save final checkpoint

In [ ]:
final_path = CKPT_DIR / 'multihead_final.pt'
torch.save({
    'model':   model.state_dict(),
    'cfg':     CFG,
    'norm': {
        'node_mean':   node_mean.tolist(), 'node_std':  node_std.tolist(),
        'out_mean':    out_mean.tolist(),  'out_std':   out_std.tolist(),
        'output_cols': OUTPUT_COLS,
    },
    'final_train_loss': train_losses[-1],
    'final_val_loss':   val_losses[-1],
    'per_head_train': {k: head_train[k][-1] for k in head_train},
    'n_cases':     len(all_cases),
    'data_source': 'reactingFoam_openfoam_80cases',
}, final_path)

print(f'Saved: {final_path}')
print(f'Final train loss (weighted): {train_losses[-1]:.4f}')
print(f'Final val loss (mean):       {val_losses[-1]:.4f}')
print('Per-head train loss (last epoch):')
for k in ('flow', 'heat', 'species'):
    print(f'  {k:8s}: {head_train[k][-1]:.4f}')
print('\nReady for 06_guardrail_engine.ipynb')